# CYR-GPU-001 — Cymek GPU research tournament (preregistered)

Run cells in order: **CELL 0 → CELL 1 → CELL 2**. Use a **GPU runtime**
(Runtime → Change runtime type → GPU). Target 90–150 min, hard stop 180.
Do not edit thresholds, seeds, or budgets: they are preregistered and
hash-bound (CELL 0 aborts on mismatch).

In [ ]:
# CELL 0 — clone exact branch, verify prereg SHAs, env check, smoke
import hashlib, json, subprocess, sys

REPO = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
BRANCH = "cymek-500m-readiness"
WORK = "/content/An-Ra-the-new-AGI"

print(subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "50",
                      REPO, WORK], capture_output=True, text=True).stderr[-500:])
%cd /content/An-Ra-the-new-AGI
!git rev-parse HEAD
!pip install -q pytest tokenizers psutil

from pathlib import Path
prereg = json.loads(Path("docs/cymek/experiments/CYR-GPU-001/PREREGISTRATION.json").read_text())
ok = True
for rel, expected in prereg["code_sha256"].items():
    actual = hashlib.sha256(Path(rel).read_bytes()).hexdigest()
    match = actual == expected
    ok = ok and match
    print(("OK " if match else "MISMATCH ") + rel)
assert ok, "preregistered code mismatch: refusing to run"
print("prereg", prereg["sha256"][:12], "experiment", prereg["experiment_id"])

import torch
assert torch.cuda.is_available(), "no GPU runtime: choose Runtime → GPU"
props = torch.cuda.get_device_properties(0)
print("gpu:", props.name, "vram_gb:", round(props.total_memory / 2**30, 1),
      "torch:", torch.__version__, "bf16:", props.major >= 8)

!python -m pytest tests/test_v5_cyr_tournament.py tests/test_v5_contracts.py tests/test_v5_complete_answer.py tests/test_v5_training.py tests/test_v5_data.py tests/test_v5_data_pipeline.py tests/test_v5_production_backend.py tests/test_v5_stream_resume.py -q 2>&1 | tail -3
!python -m anra_v5.cyr_execute --mode smoke --out /content/CYR-SMOKE
# Full entry suite on Colab hardware (exact-head self-check excluded:
# the repo receipt refreshes from Colab evidence next cycle).
!python -m pytest tests/test_production_entry.py -q --deselect tests/test_production_entry.py::test_exact_head_test_receipt 2>&1 | tail -3
print("CELL 0 COMPLETE")


In [ ]:
# CELL 1 — full preregistered tournament (streams progress, ~90-150 min)
import os
os.environ["COLAB_GPU"] = "1"
%cd /content/An-Ra-the-new-AGI
!python -m anra_v5.cyr_execute --mode full --prereg docs/cymek/experiments/CYR-GPU-001/PREREGISTRATION.json --repo . --out /content/CYR-GPU-001 --stages s0,s1,s2,s3,s4
print("CELL 1 COMPLETE")


In [ ]:
# CELL 2 — verify bundle, print decision, download ZIP
import json
from pathlib import Path
out = Path("/content/CYR-GPU-001")
manifest = json.loads((out / "SESSION_MANIFEST.json").read_text())
print("files:", manifest["files"])
decision = json.loads((out / "DECISION.json").read_text())
print(json.dumps({k: decision[k] for k in ("experiment", "stages", "gates", "claim_ladder")}, indent=2))
print("promotion_bar:", json.dumps(decision["promotion_bar"], indent=2))
from google.colab import files
files.download(str(out / "CYMEK_GPU_RESEARCH_RESULTS.zip"))
print("Return CYMEK_GPU_RESEARCH_RESULTS.zip to the operator.")
